In [ ]:
import galsim
import matplotlib.pyplot as plt
import numpy as np
import pyccl
from torch.distributions import Normal
# import torch

In [ ]:
dist = Normal(torch.tensor([0.8159]), torch.tensor([0.001]))
sigma_8 = dist.sample().item()
print(sigma_8)

In [ ]:
# cosmology = pyccl.cosmology.Cosmology(sigma8=sigma_8,
#                                       Omega_c=0.25,
#                                       Omega_b=0.0486,
#                                       h=67.7, # order of magnitude is wrong
#                                       n_s=1,
# )
cosmology = pyccl.cosmology.CosmologyVanillaLCDM()

In [ ]:
cosmology.compute_linear_power()
power_spectrum = cosmology.get_linear_power()

In [ ]:
print(power_spectrum)

In [ ]:
# power_spectrum.get_spline_arrays[2] returns P(k, a), but we want k for a fixed a

power_spectrum_vals_pk = power_spectrum.get_spline_arrays()[2][0]
power_spectrum_vals_k = np.exp(power_spectrum.get_spline_arrays()[1])

power_spectrum_vals_pk = np.array([power_spectrum_vals_pk]).T
power_spectrum_vals_k = np.array([power_spectrum_vals_k]).T
print(power_spectrum_vals_pk.shape)

with open("./sample_ps_vals.txt", "w") as f:
    print(power_spectrum_vals_k.shape, power_spectrum_vals_pk.shape)
    x = np.hstack((power_spectrum_vals_k, power_spectrum_vals_pk))
    print(x.shape)
    np.savetxt(f, x)

In [ ]:
lookup = galsim.LookupTable(
    x=power_spectrum_vals_k, f=power_spectrum_vals_pk, x_log=False, f_log=False
)

In [ ]:
galsim_ps = galsim.PowerSpectrum(lookup, units=galsim.megaparsecs)

In [ ]:
grid_size = 10.0  # Define the total grid extent, in degrees
ngrid = 20  # Define the number of grid points in each dimension: (ngrid x ngrid)
n_ell = 15

g1, g2, kappa = galsim_ps.buildGrid(
    grid_spacing=grid_size / ngrid, ngrid=ngrid, units=galsim.degrees, get_convergence=True
)

fig, (one, two) = plt.subplots(nrows=1, ncols=2)
_ = one.imshow(g1)
_ = two.imshow(g2)
print(kappa)